<a href="https://colab.research.google.com/github/indahkhairunnisah-afk/Tugas-2-Teknik-Pengambilan-Sampel-dan-Data-Wrangling/blob/main/MID_TPSDW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install requests beautifulsoup4

*syntax ini memrintahkan menyiapkan Python agar mampu melakukan pengambilan data (data acquisition) melalui protokol HTTP dan analisis konten web (web content parsing) secara terintegrasi.*

In [3]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import base64

*Python sedang “mempersiapkan peralatan” untuk bekerja. Requests agar bisa berkomunikasi dengan web, lalu menghadirkan BeautifulSoup sebagai alat untuk mengurai struktur HTML. Setelah itu, Python menyiapkan json untuk mengelola data dalam format JavaScript Object Notation, serta pandas sebagai perangkat analisis dan pengolahan data dalam bentuk tabel. Terakhir, ia membawa base64 untuk melakukan proses encoding dan decoding data biner menjadi teks.*

In [4]:
List_Android = []

for page in range(1, 4):
    url = f"https://myhartono.com/en/android/page-{page}/"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    products = soup.find_all("div", class_="ty-grid-list__item")

    for product in products:
        title = product.find("a", class_="product-title")

        # Harga asli (dicoret jika promo)
        harga_asli = product.find("span", class_="ty-strike")

        # Harga promo / harga normal
        harga_promo = product.find("span", class_="ty-price-num")

        List_Android.append({
            "title"      : title.get_text(strip=True) if title else "N/A",
            "harga_asli" : harga_asli.get_text(strip=True) if harga_asli else None,
            "harga_promo": harga_promo.get_text(strip=True) if harga_promo else None
        })

    print(f"Halaman {page} selesai — {len(products)} produk ditemukan")

print(f"\nTotal produk: {len(List_Android)}")

Halaman 1 selesai — 24 produk ditemukan
Halaman 2 selesai — 24 produk ditemukan
Halaman 3 selesai — 20 produk ditemukan

Total produk: 68


*Syntax ini melakukan web scraping untuk mengumpulkan daftar produk Android dari situs myhartono.com. Pertama, menyiapkan wadah kosong bernama List_Android. Lalu, melalui perulangan for page in range(1, 4), Python membuka halaman 1 hingga 3 secara bergantian. Setiap kali membuka halaman, ia mengirim permintaan HTTP dengan requests, kemudian mengurai isi HTML menggunakan BeautifulSoup. Dari hasil parsing, Python mencari elemen judul produk (a dengan kelas product-title) dan harga (span dengan kelas ty-price-num). Selanjutnya, setiap pasangan judul dan harga dimasukkan ke dalam List_Android sebagai dictionary. Terakhir, Python mencetak informasi bahwa halaman tertentu selesai diproses beserta jumlah produk yang ditemukan.*

In [5]:
print(List_Android[:3])

[{'title': 'SAMSUNG SMARTPHONE GALAXY A57 5G SERIES', 'harga_asli': 'Rp7.099.000', 'harga_promo': 'Rp'}, {'title': 'SAMSUNG SMARTPHONE GALAXY A37 SERIES', 'harga_asli': 'Rp6.099.000', 'harga_promo': 'Rp'}, {'title': 'SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES', 'harga_asli': 'Rp28.499.000', 'harga_promo': 'Rp'}]


*Syntax ini menyimpan data produk Android yang dikumpulkan ke dalam file JSON dengan format terstruktur dan mudah dibaca.*

In [6]:
with open("List_Android.json", "w", encoding="utf-8") as f:
    json.dump(List_Android, f, ensure_ascii=False, indent=4)

print("File List_Android.json berhasil disimpan")

File List_Android.json berhasil disimpan


 *Syntax ini membuka file JSON hasil scraping, mengubahnya kembali menjadi objek Python, lalu menampilkan sebagian kecil isi data untuk verifikasi.*

In [7]:
with open("List_Android.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(data[:3])

[{'title': 'SAMSUNG SMARTPHONE GALAXY A57 5G SERIES', 'harga_asli': 'Rp7.099.000', 'harga_promo': 'Rp'}, {'title': 'SAMSUNG SMARTPHONE GALAXY A37 SERIES', 'harga_asli': 'Rp6.099.000', 'harga_promo': 'Rp'}, {'title': 'SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES', 'harga_asli': 'Rp28.499.000', 'harga_promo': 'Rp'}]


*ode ini membuka file JSON hasil scraping, mengubahnya kembali menjadi objek Python, lalu menampilkan sebagian kecil isi data untuk verifikasi.*

In [9]:
with open("List_Android.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print("EXTRACT - Data awal:")
print(df.head())
print("Shape:", df.shape)

EXTRACT - Data awal:
                                         title    harga_asli harga_promo
0      SAMSUNG SMARTPHONE GALAXY A57 5G SERIES   Rp7.099.000          Rp
1         SAMSUNG SMARTPHONE GALAXY A37 SERIES   Rp6.099.000          Rp
2  SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES  Rp28.499.000          Rp
3  SAMSUNG SMARTPHONE GALAXY Z FLIP7 5G SERIES  Rp17.999.000          Rp
4   SAMSUNG SMARTPHONE GALAXY S25 ULTRA SERIES  Rp21.009.000          Rp
Shape: (68, 3)


*kode ini membaca file JSON hasil scraping, mengubahnya menjadi tabel pandas, lalu menampilkan contoh isi dan dimensi data sebagai tahap awal proses analisis.*

In [10]:
# Cleaning kolom title
df["title"] = df["title"].str.strip()

# Fungsi cleaning harga
def clean_harga(harga):
    if harga is None or harga == "N/A":
        return None
    harga = str(harga)
    harga = harga.replace(",", "")
    harga = harga.replace(".", "")
    harga = harga.replace("Rp", "")
    harga = harga.strip()
    return int(harga) if harga.isdigit() else None

# Cleaning harga asli dan harga promo
df["harga_asli"]  = df["harga_asli"].apply(clean_harga)
df["harga_promo"] = df["harga_promo"].apply(clean_harga)

# Isi price kosong karena promo
# Jika harga_asli kosong, gunakan harga_promo sebagai harga utama
df["price"] = df["harga_asli"].fillna(df["harga_promo"])

# Jika keduanya kosong, isi dengan keterangan
df["price"] = df["price"].fillna("Harga Tidak Tersedia")

# Tambah kolom diskon
df["selisih_harga"] = df.apply(
    lambda row: row["harga_asli"] - row["harga_promo"]
    if pd.notna(row["harga_asli"]) and pd.notna(row["harga_promo"])
    else None, axis=1
)

df["persen_diskon"] = df.apply(
    lambda row: round((row["selisih_harga"] / row["harga_asli"]) * 100, 2)
    if pd.notna(row["selisih_harga"]) and row["harga_asli"] != 0
    else 0, axis=1
)

# Tandai status produk
df["status"] = df["selisih_harga"].apply(
    lambda x: "Promo" if pd.notna(x) and x > 0 else "Normal"
)

# Hapus duplikat dan reset index
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print("TRANSFORM - Data setelah cleaning:")
print(df.head())
print("\nJumlah price kosong:", df["price"].isnull().sum())
print("Produk Normal :", len(df[df["status"] == "Normal"]))
print("Produk Promo  :", len(df[df["status"] == "Promo"]))

TRANSFORM - Data setelah cleaning:
                                         title  harga_asli harga_promo  \
0      SAMSUNG SMARTPHONE GALAXY A57 5G SERIES     7099000        None   
1         SAMSUNG SMARTPHONE GALAXY A37 SERIES     6099000        None   
2  SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES    28499000        None   
3  SAMSUNG SMARTPHONE GALAXY Z FLIP7 5G SERIES    17999000        None   
4   SAMSUNG SMARTPHONE GALAXY S25 ULTRA SERIES    21009000        None   

      price selisih_harga  persen_diskon  status  
0   7099000          None              0  Normal  
1   6099000          None              0  Normal  
2  28499000          None              0  Normal  
3  17999000          None              0  Normal  
4  21009000          None              0  Normal  

Jumlah price kosong: 0
Produk Normal : 68
Produk Promo  : 0


*kode ini membersihkan data harga, mengisi nilai kosong, menghitung diskon, memberi label status produk, lalu menampilkan ringkasan hasil transformasi*

In [11]:
# Cleaning kolom title
df["title"] = df["title"].str.strip()

# Fungsi cleaning harga
def clean_harga(harga):
    if harga is None or harga == "N/A":
        return None
    harga = str(harga)
    harga = harga.replace(",", "")
    harga = harga.replace(".", "")
    harga = harga.replace("Rp", "")
    harga = harga.strip()
    return int(harga) if harga.isdigit() else None

# Cleaning harga asli dan harga promo
df["harga_asli"]  = df["harga_asli"].apply(clean_harga)
df["harga_promo"] = df["harga_promo"].apply(clean_harga)

# Isi price kosong karena promo
# Jika harga_asli kosong, gunakan harga_promo sebagai harga utama
df["price"] = df["harga_asli"].fillna(df["harga_promo"])

# Jika keduanya kosong, isi dengan keterangan
df["price"] = df["price"].fillna("Harga Tidak Tersedia")

# Tambah kolom diskon
df["selisih_harga"] = df.apply(
    lambda row: row["harga_asli"] - row["harga_promo"]
    if pd.notna(row["harga_asli"]) and pd.notna(row["harga_promo"])
    else None, axis=1
)

df["persen_diskon"] = df.apply(
    lambda row: round((row["selisih_harga"] / row["harga_asli"]) * 100, 2)
    if pd.notna(row["selisih_harga"]) and row["harga_asli"] != 0
    else 0, axis=1
)

# Tandai status produk
df["status"] = df["selisih_harga"].apply(
    lambda x: "Promo" if pd.notna(x) and x > 0 else "Normal"
)

# Hapus duplikat dan reset index
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print("TRANSFORM - Data setelah cleaning:")
print(df.head())
print("\nJumlah price kosong:", df["price"].isnull().sum())
print("Produk Normal :", len(df[df["status"] == "Normal"]))
print("Produk Promo  :", len(df[df["status"] == "Promo"]))

TRANSFORM - Data setelah cleaning:
                                         title  harga_asli harga_promo  \
0      SAMSUNG SMARTPHONE GALAXY A57 5G SERIES     7099000        None   
1         SAMSUNG SMARTPHONE GALAXY A37 SERIES     6099000        None   
2  SAMSUNG SMARTPHONE GALAXY Z FOLD7 5G SERIES    28499000        None   
3  SAMSUNG SMARTPHONE GALAXY Z FLIP7 5G SERIES    17999000        None   
4   SAMSUNG SMARTPHONE GALAXY S25 ULTRA SERIES    21009000        None   

      price selisih_harga  persen_diskon  status  
0   7099000          None              0  Normal  
1   6099000          None              0  Normal  
2  28499000          None              0  Normal  
3  17999000          None              0  Normal  
4  21009000          None              0  Normal  

Jumlah price kosong: 0
Produk Normal : 68
Produk Promo  : 0


*kode ini membersihkan data harga, mengisi nilai kosong, menghitung diskon, memberi label status produk, lalu menampilkan ringkasan hasil transformasi.*

In [12]:
df.to_csv("List_Android.csv", index=False, encoding="utf-8")
print("File List_Android.csv berhasil disimpan")

File List_Android.csv berhasil disimpan


*Syntax ini mengubah tabel hasil scraping menjadi file CSV yang rapi, lalu memberi tahu bahwa file tersebut telah tersimpan.*